In [1]:
import os

# 1. Create the .env file
# Replace 'YOUR_NGROK_TOKEN' below with your actual token string
ngrok_token = "3AHFx7J2gwVDxeND5JfqajbiApl_4yK8cvESPsWQvimpGFLwb" 

with open(".env", "w") as f:
    f.write(f"NGROK_AUTHTOKEN={ngrok_token}\n")

print(f".env file created with token: {ngrok_token[:4]}...")

.env file created with token: 3AHF...


In [2]:
import torch
from huggingface_hub import notebook_login

print("CUDA available:", torch.cuda.is_available())
print("GPUs found:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

# Login to access Google's Gemma model
notebook_login()

CUDA available: True
GPUs found: 2
Device Name: Tesla T4


In [ ]:
# 1. Uninstall blockers (suppressed output)
!pip uninstall -y numpy jax jaxlib tensorflow bigframes vllm vllm-flash-attn torchvision torchaudio -q 2>/dev/null

# 2. Install vLLM 0.7.0 + matching transformers + AWQ support
print("⏳ Installing vLLM 0.7.0 + deps (this takes ~3-5 min)...")
!pip install -q vllm==0.7.0 torchvision "transformers>=4.47,<4.49" autoawq "huggingface_hub>=0.23,<0.27" --extra-index-url https://download.pytorch.org/whl/cu121

# 3. Install networking tools
!pip install -q fastapi uvicorn pyngrok python-dotenv

print("✅ Installation complete.")
print("⚠️  YOU MUST RESTART THE SESSION NOW (Runtime → Restart session)")
print("⚠️  Then skip this cell and run from Cell 4 onwards.")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: bigframes 2.35.0
Uninstalling bigframes-2.35.0:
  Successfully uninstalled bigframes-2.35.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
⏳ Installing vLLM 0.7.0...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

In [4]:
# Verify versions (run AFTER session restart — do NOT run Cell 3 again)
import transformers
print(f"Transformers: {transformers.__version__}")
assert transformers.__version__ >= "4.47", f"Need transformers >=4.47, got {transformers.__version__}"
print("✅ Ready to run server.")

Transformers: 5.0.0
✅ Ready to run server.


In [ ]:
# Pre-download model weights (visible progress bar, separate from vLLM startup)
import os, subprocess, time

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"
CACHE_DIR = os.path.expanduser("~/.cache/huggingface/hub")
model_dir_name = "models--" + MODEL_ID.replace("/", "--")
cached = os.path.exists(os.path.join(CACHE_DIR, model_dir_name, "snapshots"))

if cached:
    print(f"✅ {MODEL_ID} already in cache — skipping download.")
else:
    # Fix huggingface_hub first (Kaggle pre-installs an incompatible version)
    print("Fixing huggingface_hub version...")
    subprocess.run('pip install -q "huggingface_hub>=0.23,<0.27"', shell=True, check=True)

    print(f"⏳ Downloading {MODEL_ID} (~8 GB)...")
    start = time.time()
    # Run in subprocess to avoid stale module cache in this kernel
    rc = subprocess.run([
        "python", "-c",
        f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}', cache_dir='{CACHE_DIR}')"
    ]).returncode
    if rc != 0:
        raise RuntimeError("Model download failed")
    elapsed = time.time() - start
    print(f"✅ Download complete in {elapsed:.0f}s")

In [ ]:
import os
import re
import time
import signal
import shutil
import requests
import subprocess
import threading

# ==========================================
# 0. PRE-FLIGHT: CLEAN UP EVERYTHING
# ==========================================
subprocess.run("pkill -9 -f 'vllm serve' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'vllm.entrypoints' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'ray::' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'raylet' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'gcs_server' 2>/dev/null", shell=True)
subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True)
time.sleep(2)

# Clean stale state
if os.path.exists("/tmp/ray"):
    shutil.rmtree("/tmp/ray", ignore_errors=True)
for f in os.listdir("/dev/shm"):
    try: os.unlink(os.path.join("/dev/shm", f))
    except: pass

# Clean HF lock files
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
for root, dirs, files in os.walk(hf_cache):
    for f in files:
        if f.endswith(".lock"):
            try: os.unlink(os.path.join(root, f))
            except: pass

subprocess.run(["fuser", "-k", "/dev/nvidia0"], capture_output=True)
subprocess.run(["fuser", "-k", "/dev/nvidia1"], capture_output=True)
time.sleep(2)

# ── KEY DIAGNOSTIC: Check available RAM ──
print("=" * 60)
ram = subprocess.check_output("free -h", shell=True).decode()
print(ram)
# Check for recent OOM kills
dmesg = subprocess.run("dmesg -T 2>/dev/null | grep -i -E 'oom|killed|out of memory' | tail -5",
                        shell=True, capture_output=True, text=True).stdout.strip()
if dmesg:
    print(f"⚠️  Recent OOM kills detected:\n{dmesg}")
else:
    print("✓ No recent OOM kills in dmesg")
print("=" * 60)

gpu_check = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
    text=True
).strip()
print(f"GPU memory: {gpu_check}")

# ==========================================
# 1. MODEL + VLLM SETTINGS
# ==========================================
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"
VLLM_PORT = 8000

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    gpu_count = max(len(result.stdout.strip().splitlines()), 1)
except Exception:
    gpu_count = 1

def get_gpu_mem_gb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True, timeout=5
        ).strip()
        return [round(int(x) / 1024, 1) for x in out.splitlines()]
    except Exception:
        return []

def build_cmd(tp_size, backend="mp"):
    cmd = [
        "vllm", "serve", MODEL_ID,
        "--host", "0.0.0.0",
        "--port", str(VLLM_PORT),
        "--tensor-parallel-size", str(tp_size),
        "--gpu-memory-utilization", "0.90",
        "--dtype", "half",
        "--max-model-len", "8192" if tp_size > 1 else "4096",
        "--max-num-seqs", "4",
        "--enforce-eager",
        "--quantization", "awq",
        "--trust-remote-code",
        "--distributed-executor-backend", backend,
    ]
    if tp_size > 1:
        cmd += ["--disable-custom-all-reduce"]
    return cmd

def start_and_wait(tp_size, backend="mp", timeout=600):
    """Start vLLM and wait for it to become ready. Returns (proc, success)."""
    label = f"TP={tp_size}, backend={backend}"
    print(f"\n▶ Starting vLLM with {label}...")

    cmd = build_cmd(tp_size, backend)
    with open("vllm_qwen.log", "w") as f:
        proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, preexec_fn=os.setsid)

    print(f"{'Time':>6}  {'GPU0':>8}  {'GPU1':>8}  Status")
    print(f"{'─'*6}  {'─'*8}  {'─'*8}  {'─'*45}")

    start = time.time()
    last_mem = []
    stuck_count = 0

    while True:
        elapsed = int(time.time() - start)

        if proc.poll() is not None:
            print(f"\n❌ vLLM crashed after {elapsed}s:")
            print(subprocess.check_output(["tail", "-n", "30", "vllm_qwen.log"]).decode())
            return proc, False

        try:
            r = requests.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=3)
            if r.status_code == 200:
                mem = get_gpu_mem_gb()
                print(f"\n✅ vLLM ready in {elapsed}s  |  VRAM: {mem}")
                return proc, True
        except Exception:
            pass

        mem = get_gpu_mem_gb()
        g0 = f"{mem[0]:>6} GB" if len(mem) > 0 else "   ?   "
        g1 = f"{mem[1]:>6} GB" if len(mem) > 1 else "   N/A "

        try:
            tail = subprocess.check_output(["tail", "-n", "1", "vllm_qwen.log"]).decode().strip()
            short = tail.split("] ", 1)[-1] if "] " in tail else tail
            status = short[:45]
        except Exception:
            status = "..."

        # Detect dead worker (THE KEY CHECK)
        if elapsed > 40 and mem == last_mem and mem and all(m > 0 for m in mem):
            stuck_count += 1
        else:
            stuck_count = 0
        last_mem = mem

        if stuck_count >= 6 and tp_size > 1:  # 60s stuck
            # Check if worker process is alive
            ps = subprocess.check_output(
                f"pgrep -f 'from multiprocessing' --count 2>/dev/null || echo 0",
                shell=True).decode().strip()
            worker_alive = int(ps) > 0 if ps.isdigit() else False

            if not worker_alive:
                print(f"\n💀 Worker process DIED! Main process is stuck waiting.")
                # Check OOM
                oom = subprocess.run(
                    "dmesg -T 2>/dev/null | grep -i 'oom\\|killed' | tail -3",
                    shell=True, capture_output=True, text=True).stdout.strip()
                if oom:
                    print(f"   OOM evidence: {oom}")
                else:
                    print("   No OOM in dmesg — likely CUDA error or segfault in worker")
                try:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                except Exception:
                    pass
                time.sleep(2)
                subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True)
                return proc, False

        if stuck_count >= 12 and tp_size == 1:
            print(f"\n⚠️  Stuck for {stuck_count*10}s with TP=1:")
            print(subprocess.check_output(["tail", "-n", "20", "vllm_qwen.log"]).decode())

        print(f"{elapsed:>5}s  {g0}  {g1}  {status}")

        if elapsed > timeout:
            print(f"\n❌ Timed out after {elapsed}s")
            print(subprocess.check_output(["tail", "-n", "20", "vllm_qwen.log"]).decode())
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except Exception:
                pass
            return proc, False
        time.sleep(10)

# ==========================================
# 2. TRY TO START (with automatic fallback)
# ==========================================

# Attempt 1: TP=2 with multiprocessing (default)
vllm_proc, ok = start_and_wait(gpu_count, backend="mp", timeout=300)

if not ok and gpu_count > 1:
    print("\n" + "="*60)
    print("⚡ Attempt 2: TP=2 with Ray backend (different worker model)...")
    print("="*60)
    subprocess.run(["fuser", "-k", "/dev/nvidia0"], capture_output=True)
    subprocess.run(["fuser", "-k", "/dev/nvidia1"], capture_output=True)
    subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True)
    time.sleep(3)
    vllm_proc, ok = start_and_wait(gpu_count, backend="ray", timeout=600)

if not ok and gpu_count > 1:
    print("\n" + "="*60)
    print("⚡ Attempt 3: TP=1 on single GPU (reduced context but avoids TP hang)")
    print("="*60)
    subprocess.run(["fuser", "-k", "/dev/nvidia0"], capture_output=True)
    subprocess.run(["fuser", "-k", "/dev/nvidia1"], capture_output=True)
    subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True)
    time.sleep(3)
    vllm_proc, ok = start_and_wait(1, backend="mp", timeout=600)

if not ok:
    raise RuntimeError("All vLLM start attempts failed. Check logs above.")

# ==========================================
# 3. CLOUDFLARE TUNNEL
# ==========================================
cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{VLLM_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

public_url = {"value": None}

def _watch_cloudflared():
    for line in cf_proc.stdout:
        m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if m and public_url["value"] is None:
            public_url["value"] = m.group(0)
        print(line.rstrip())

t = threading.Thread(target=_watch_cloudflared, daemon=True)
t.start()

for _ in range(80):
    if public_url["value"]:
        break
    time.sleep(0.5)

if not public_url["value"]:
    raise RuntimeError("Could not detect Cloudflare URL from logs")

print(f"\n{'='*60}")
print(f"PUBLIC URL : {public_url['value']}")
print(f"Endpoint   : {public_url['value']}/v1/chat/completions")
print(f"Model      : {MODEL_ID}")
print(f"{'='*60}")

Detected 2 GPU(s). Starting vLLM with TP=2...
....................................................................................................................................................................................

TimeoutError: Timed out waiting for vLLM to start

In [6]:
!tail -n 80 vllm_qwen.log

INFO 04-12 09:01:39 __init__.py:183] Automatically detected platform cuda.
INFO 04-12 09:01:40 api_server.py:835] vLLM API server version 0.7.0
INFO 04-12 09:01:40 api_server.py:836] args: Namespace(subparser='serve', model_tag='Qwen/Qwen2.5-14B-Instruct-AWQ', config='', host='0.0.0.0', port=8000, uvicorn_log_level='info', allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plugin='', model='Qwen/Qwen2.5-14B-Instruct-AWQ', task='auto', tokenizer=None, skip_tokenizer_init=False, revision=None, code_revision=None, tokenizer_revision=None,

In [ ]:
print(subprocess.check_output(["tail", "-n", "100", "vllm_qwen.log"]).decode())

# **Clean Up**

In [ ]:
vllm_proc.terminate()
vllm_proc.wait()
print("vLLM stopped, exit code:", vllm_proc.returncode)
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Nuclear option: kill everything + clean Ray + /dev/shm
import subprocess, os, shutil, time

subprocess.run("pkill -9 -f 'vllm serve' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'vllm.entrypoints' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'ray::' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'raylet' 2>/dev/null", shell=True)
subprocess.run("pkill -9 -f 'gcs_server' 2>/dev/null", shell=True)
subprocess.run(["fuser", "-k", "/dev/nvidia0"], capture_output=True)
subprocess.run(["fuser", "-k", "/dev/nvidia1"], capture_output=True)
subprocess.run("fuser -k 8000/tcp 2>/dev/null", shell=True)
time.sleep(2)

# Clean Ray state (main culprit for TP worker deadlocks after restarts)
if os.path.exists("/tmp/ray"):
    shutil.rmtree("/tmp/ray", ignore_errors=True)
    print("✓ /tmp/ray cleaned")

# Clean /dev/shm
for f in os.listdir("/dev/shm"):
    try: os.unlink(os.path.join("/dev/shm", f))
    except: pass

print(subprocess.check_output(["nvidia-smi"]).decode())

In [4]:
import os, signal, gc, ctypes, subprocess, time

# 1. Kill vLLM + all its child workers (tensor-parallel spawns subprocesses)
try:
    os.killpg(os.getpgid(vllm_proc.pid), signal.SIGKILL)
except Exception:
    try:
        vllm_proc.kill()
        vllm_proc.wait(timeout=5)
    except Exception:
        pass
print("vLLM killed")

# 2. Kill cloudflared
try:
    cf_proc.kill()
    cf_proc.wait(timeout=5)
except Exception:
    pass
print("cloudflared killed")

# 3. Nuclear: kill anything still on the GPUs
subprocess.run(["fuser", "-k", "/dev/nvidia0"], capture_output=True)
subprocess.run(["fuser", "-k", "/dev/nvidia1"], capture_output=True)
time.sleep(2)

# 4. Clear Python-side CUDA state
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect()

# 5. Trim glibc malloc arenas (frees RSS back to OS)
ctypes.CDLL("libc.so.6").malloc_trim(0)

# 6. Drop references
for name in ["vllm_proc", "cf_proc", "model", "tokenizer", "public_url"]:
    if name in globals():
        del globals()[name]

gc.collect()

# 7. Verify
print(subprocess.check_output(["nvidia-smi"]).decode())

vLLM killed
cloudflared killed
Sun Apr 12 08:59:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------